In [0]:
from pyspark.sql import Row

catalog_name = "ecommerce"

country_state_map = {
    "India":          {"IN": "APAC"},
    "United States":  {"US": "AMER"},
    "United Kingdom": {"GB": "EMEA"},
    "UAE":            {"AE": "EMEA"},
    "Singapore":      {"SG": "APAC"},
    "Canada":         {"CA": "AMER"},
    "Australia":      {"AU": "APAC"},
}

rows = []
for country, states in country_state_map.items():
    for state_code, region in states.items():
        rows.append(Row(country=country, state=state_code, region=region))

rows[:10]

In [0]:
df_region_mapping = spark.createDataFrame(rows)
display(df_region_mapping)

In [0]:
df_silver = spark.table(f"{catalog_name}.silver.slv_customers")
display(df_silver.limit(5))

In [0]:
df_gold = df_silver.join(df_region_mapping, on="state", how="left")
df_gold = df_gold.fillna({"region": "Other"})
display(df_gold.limit(5))

In [0]:
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.gold.gld_dim_customers")

In [0]:
import pyspark.sql.functions as F

df_silver = spark.table(f"{catalog_name}.silver.slv_calendar")
display(df_silver.limit(5))

In [0]:
df_gold = df_silver.withColumn("date_id", F.date_format(F.col("date"), "yyyyMMdd").cast("int"))
df_gold = df_gold.withColumn("month_name", F.date_format(F.col("date"), "MMMM"))
df_gold = df_gold.withColumn("is_weekend", F.when(F.col("day_name").isin("Saturday", "Sunday"), 1).otherwise(0))
df_gold = df_gold.withColumn("week", F.concat_ws("-", F.lit("Week"), F.weekofyear(F.col("date")), F.col("year")))
display(df_gold.limit(5))

In [0]:
desired_columns_order = ["date_id", "date", "year", "month_name", "day_name", "is_weekend", "quarter", "week", "_ingested_at", "_source_file"]
df_gold = df_gold.select(desired_columns_order)
display(df_gold.limit(5))

In [0]:
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.gold.gld_dim_date")